# 1. Logistic Regression
### Binary and Multiclass Classification

**Logistic Regression** is a fundamental linear classification algorithm. Despite its name, it is used for predicting categorical outcomes (binary or multiclass) rather than continuous values.

#### Key Concepts:
1. **Odds and Log-Odds (Logit)**:
   $$\text{logit}(p) = \ln\left(\frac{p}{1-p}\right) = w_0 + w_1 x_1 + \dots + w_n x_n$$
2. **Sigmoid Activation** (for Binary Classification):
   $$\sigma(z) = \frac{1}{1 + e^{-z}}$$
   Maps any real-valued number into a probability value between 0 and 1.
3. **Softmax Function** (for Multiclass Classification):
   $$P(Y = k | X) = \frac{e^{w_k^T X}}{\sum_{j=1}^{K} e^{w_j^T X}}$$
4. **Loss Function**: Binary Cross-Entropy (Log Loss) or Multiclass Cross-Entropy optimized using solvers like L-BFGS, Saga, or Liblinear.


In [ ]:
# Step 1: Import Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer, load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay


### Part A: Binary Classification (Breast Cancer Dataset)


In [ ]:
# Load binary dataset
cancer = load_breast_cancer()
X_bin = cancer.data
y_bin = cancer.target

# Split data
X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X_bin, y_bin, test_size=0.25, random_state=42, stratify=y_bin
)

# Feature scaling (important for Logistic Regression with regularization)
scaler_b = StandardScaler()
X_train_b_scaled = scaler_b.fit_transform(X_train_b)
X_test_b_scaled = scaler_b.transform(X_test_b)

# Train Binary Logistic Regression
binary_model = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
binary_model.fit(X_train_b_scaled, y_train_b)

# Evaluation
y_pred_b = binary_model.predict(X_test_b_scaled)
y_prob_b = binary_model.predict_proba(X_test_b_scaled)[:, 1]

print("=== Binary Classification Results ===")
print(f"Accuracy: {accuracy_score(y_test_b, y_pred_b):.4f}")
print("
Classification Report:
", classification_report(y_test_b, y_pred_b, target_names=cancer.target_names))


### Part B: Multiclass Classification (Iris Dataset)


In [ ]:
# Load multiclass dataset
iris = load_iris()
X_multi = iris.data
y_multi = iris.target

# Split data
X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_multi, y_multi, test_size=0.25, random_state=42, stratify=y_multi
)

scaler_m = StandardScaler()
X_train_m_scaled = scaler_m.fit_transform(X_train_m)
X_test_m_scaled = scaler_m.transform(X_test_m)

# Multiclass with Multinomial / Softmax
multi_model = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=1000, random_state=42)
multi_model.fit(X_train_m_scaled, y_train_m)

y_pred_m = multi_model.predict(X_test_m_scaled)
print("=== Multiclass Classification Results ===")
print(f"Accuracy: {accuracy_score(y_test_m, y_pred_m):.4f}")
print("
Classification Report:
", classification_report(y_test_m, y_pred_m, target_names=iris.target_names))


### Part C: Decision Boundary Visualization (2 Features)


In [ ]:
# Visualizing 2D decision boundary on first 2 features of Iris
X_2d = X_train_m_scaled[:, :2]
y_2d = y_train_m

vis_model = LogisticRegression(multi_class='multinomial', solver='lbfgs', random_state=42)
vis_model.fit(X_2d, y_2d)

x_min, x_max = X_2d[:, 0].min() - 1, X_2d[:, 0].max() + 1
y_min, y_max = X_2d[:, 1].min() - 1, X_2d[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02),
                     np.arange(y_min, y_max, 0.02))

Z = vis_model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.coolwarm)
scatter = plt.scatter(X_2d[:, 0], X_2d[:, 1], c=y_2d, cmap=plt.cm.coolwarm, edgecolors='k')
plt.xlabel("Feature 1 (Standardized)")
plt.ylabel("Feature 2 (Standardized)")
plt.title("Logistic Regression Multiclass Decision Boundaries")
plt.legend(handles=scatter.legend_elements()[0], labels=list(iris.target_names))
plt.show()


### Summary
- **Strengths**: Interpretable coefficients, probabilistic predictions, efficient to train, low risk of overfitting with regularization ($L_1$/$L_2$).
- **Weaknesses**: Assumes linear relationships between features and log-odds, sensitive to outliers and collinearity.
